# GraphRAG-Based Text Retrieval System

## Project Overview
This project implements a Graph Retrieval-Augmented Generation (GraphRAG) system that combines a Knowledge Graph (Neo4j) and a Vector Database (ChromaDB) with a Large Language Model (Gemma) to answer user questions.

Unlike traditional Retrieval-Augmented Generation (RAG), GraphRAG utilizes structured relationships between entities to improve reasoning and provide more informative answers.

## Technologies Used
1.Python
2.LangChain
3.Neo4j
4.ChromaDB
5.Sentence Transformers
6.Ollama
7.Gemma 3

## Project Objective

The objective of this project is to demonstrate how structured knowledge graphs and semantic vector retrieval can work together to enhance question answering systems.



# Table of Contents

1. Introduction
2. Problem Statement
3. What is RAG?
4. Limitations of Traditional RAG
5. What is GraphRAG?
6. Project Architecture
7. Technology Stack
8. Project Workflow
9. Configuration
10. Document Loading
11. Document Chunking
12. Embedding Generation
13. Vector Database (ChromaDB)
14. Entity Extraction
15. Knowledge Graph Construction
16. Graph Search
17. Hybrid Search
18. Answer Generation
19. Future Scope
20. Conclusion




# 1. Introduction

Large Language Models (LLMs) have significantly improved the ability of computers to understand and generate natural language. However, these models possess two important limitations:

1. They cannot access private or newly added documents without additional mechanisms.
2. They may generate inaccurate or hallucinated information.

Retrieval-Augmented Generation (RAG) addresses these issues by retrieving relevant document chunks before generating an answer.

Graph Retrieval-Augmented Generation (GraphRAG) extends this idea by incorporating structured knowledge graphs. Instead of relying only on semantic similarity, GraphRAG also understands relationships between entities, enabling better reasoning and more accurate responses.

# 2. Problem Statement

Traditional Retrieval-Augmented Generation systems retrieve information based only on semantic similarity.

Although effective, they often fail to capture structured relationships between concepts.

For example:

Machine Learning → Artificial Intelligence → Computer Science

A traditional vector search may retrieve only one of these concepts.

GraphRAG explicitly stores these relationships inside a Knowledge Graph, enabling the system to traverse connected entities and provide richer contextual answers.

The objective of this project is to combine:

- Semantic Retrieval using ChromaDB
- Structured Retrieval using Neo4j
- Answer Generation using Gemma

into a unified GraphRAG pipeline.

# 3. What is Retrieval-Augmented Generation (RAG)?

Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with external knowledge before generating an answer.

Instead of relying only on the knowledge learned during training, a RAG system retrieves relevant information from external documents and includes that information in the prompt sent to the language model.

The RAG pipeline generally consists of the following steps:

1. User asks a question.
2. The question is converted into an embedding.
3. Similar document chunks are retrieved from a vector database.
4. The retrieved chunks are added to the prompt.
5. The LLM generates the final answer.

This approach reduces hallucinations and enables the system to answer questions based on private or domain-specific documents.

# 4. Limitations of Traditional RAG

Although Retrieval-Augmented Generation significantly improves the quality of responses, it still has several limitations.

### 1. No Understanding of Relationships

A vector database retrieves information based on semantic similarity, but it does not explicitly know how different entities are connected.

For example,

Artificial Intelligence

↓

Machine Learning

↓

Deep Learning

Although these concepts are related, a vector database stores them as independent text chunks rather than as connected knowledge.

---

### 2. Limited Multi-hop Reasoning

Questions requiring multiple reasoning steps are difficult to answer because the retrieved chunks may not contain all the required information.

---

### 3. Duplicate Information

Multiple retrieved chunks may contain repeated information, making the context unnecessarily large.

---

### 4. Lack of Explainability

Traditional RAG cannot explain why two concepts are related because relationships are never stored explicitly.

# 5. What is Graph Retrieval-Augmented Generation (GraphRAG)?

Graph Retrieval-Augmented Generation (GraphRAG) extends the traditional RAG architecture by introducing a Knowledge Graph.

Instead of relying only on semantic similarity, GraphRAG stores entities and their relationships inside a graph database.

When a user asks a question, the system retrieves information from two different sources:

1. The Knowledge Graph (Neo4j), which provides structured relationships.
2. The Vector Database (ChromaDB), which provides semantically similar document chunks.

The retrieved information from both sources is combined and provided to the Large Language Model, enabling more accurate and context-aware responses.

GraphRAG is particularly useful for applications that require reasoning over connected information such as healthcare, law enforcement, cybersecurity, finance, and scientific research.

# 6. Project Architecture

The GraphRAG system consists of five major stages:

1. Document Processing
2. Knowledge Graph Construction
3. Vector Database Construction
4. Hybrid Retrieval
5. Answer Generation

Each stage performs a specific task, and together they form the complete GraphRAG pipeline.

| File | Purpose |
|------|---------|
| `config.py` | Loads models and database connections. |
| `ingest.py` | Loads and chunks the document. |
| `entity_extractor.py` | Extracts entities using Gemma. |
| `entity_to_graph.py` | Stores entities in Neo4j. |
| `relationship_extractor.py` | Extracts relationships. |
| `graph_builder.py` | Builds the Knowledge Graph. |
| `graph_search.py` | Searches Neo4j. |
| `hybrid_search.py` | Combines Graph and Vector Search. |
| `answer_generator.py` | Generates the final answer. |

## Overall Working of the Proposed GraphRAG System

**Step 1:** Load the input document into the system.

**Step 2:** Split the document into smaller text chunks for efficient processing.

**Step 3:** Generate embeddings for each chunk using the Sentence Transformer model.

**Step 4:** Store the generated embeddings in ChromaDB for semantic vector retrieval.

**Step 5:** Extract important entities from the document using the Gemma 3 LLM.

**Step 6:** Extract relationships between the identified entities.

**Step 7:** Store the entities as nodes and relationships as edges in Neo4j to build the Knowledge Graph.

**Step 8:** Accept the user's question as input.

**Step 9:** Extract the main entity from the user's question.

**Step 10:** Perform Graph Search in Neo4j to retrieve related entities and relationships.

**Step 11:** Convert the user's question into an embedding and perform Vector Search in ChromaDB.

**Step 12:** Combine the Graph Context and Vector Context into a single prompt.

**Step 13:** Send the combined context to the Gemma 3 LLM.

**Step 14:** Generate and display the final answer to the user.

# 7.Configuration(01_config.py)

The `01_config.py` file initializes all the shared resources required throughout the GraphRAG system. Instead of creating new connections in every module, the models and database connections are loaded once and reused across the project.

## 7.1 : Import Required Libraries

Import all the libraries required for the language model(OllamaLLM), embedding (SentenceTransformer), Neo4j, and ChromaDB.

In [2]:
from langchain_ollama import OllamaLLM
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
import chromadb

## 7.2 : Load the Language Model (Gemma)

The Large Language Model (LLM) is responsible for understanding natural language and generating intelligent responses. In this project, the Gemma 3 model is loaded using Ollama. It is used for entity extraction, relationship extraction, and final answer generation.

In [3]:
from langchain_ollama import OllamaLLM

print("Loading Gemma...")

llm = OllamaLLM(model="gemma3")

print("Gemma Ready")

Loading Gemma...
Gemma Ready


## 7.3 Load the Embedding Model

The embedding model converts textual data into high-dimensional numerical vectors. These vectors capture the semantic meaning of the text, allowing the system to retrieve relevant information based on meaning rather than exact keyword matching.

In this project, the `all-MiniLM-L6-v2` Sentence Transformer model is used because it is lightweight, efficient, and suitable for semantic search applications.

In [4]:
print("Loading Embedding Model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Ready")

Loading Embedding Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 880.06it/s]


Embedding Model Ready


## 7.4 Connect to Neo4j

Neo4j is used as the graph database to store entities and relationships extracted from the documents. The application establishes a connection using the Bolt protocol, which enables efficient communication with the database.

In [5]:
URI = "bolt://172.26.0.1:7687"

USERNAME = "neo4j"

PASSWORD = "********"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

print("Neo4j Connected")

Neo4j Connected


## 7.5 Connect to ChromaDB

ChromaDB is used as the vector database for storing document embeddings. It enables semantic retrieval by comparing the embedding of a user's query with the stored document vectors.

#### client = chromadb.PersistentClient
PersistentClient creates a connection to a persistent ChromaDB database.
Unlike an in-memory database, a persistent database stores data permanently on disk.
The returned client object is used to perform all operations such as creating collections, storing embeddings, and searching.



In [6]:
client = chromadb.PersistentClient(
    path="chroma_db"
)
print("ChromaDB Connected")

ChromaDB Connected


## 8.1 Read the Document(02_ingest.py)

The first step in the ingestion pipeline is to load the input document into memory. The document is read as plain text using UTF-8 encoding, ensuring that all characters are interpreted correctly. This text serves as the input for chunking and embedding generation.

In [7]:
with open(
    "sample.txt",
    "r",
    encoding="utf-8"
) as file:
    text = file.read()

print("=" * 50)
print("ORIGINAL DOCUMENT")
print("=" * 50)
print(text)
print(type(text))

ORIGINAL DOCUMENT
Artificial Intelligence is a branch of Computer Science.

Machine Learning is a subset of Artificial Intelligence.

Deep Learning is a subset of Machine Learning.

Neo4j is a graph database.

GraphRAG combines graph databases and large language models.

<class 'str'>


## 8.2 Document Chunking(02_ingest.py)

The loaded document is divided into smaller overlapping chunks using the Recursive Character Text Splitter. Chunking improves semantic retrieval by ensuring that each embedding represents a manageable portion of the document while preserving context through overlap.



In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#for checking the module being imported or not
print("Import Successful")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
#split the entire document into chunks
chunks = splitter.split_text(text)
#for maintaining a proper format of output
print("\n" + "=" * 50)
print("DOCUMENT CHUNKS")
print("=" * 50)
#here the enumerate helps in adding number with the chunks
for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}")#chunk number
    print("-" * 30)#for mainataining the format of output
    print(chunk)#the respective chunk is printed
print("Document Length:", len(text))#entire text document length
print("Total Chunks:", len(chunks))#the length of the chunk

Import Successful

DOCUMENT CHUNKS

Chunk 1
------------------------------
Artificial Intelligence is a branch of Computer Science.

Chunk 2
------------------------------
Machine Learning is a subset of Artificial Intelligence.

Chunk 3
------------------------------
Deep Learning is a subset of Machine Learning.

Neo4j is a graph database.

Chunk 4
------------------------------
GraphRAG combines graph databases and large language models.
Document Length: 253
Total Chunks: 4


### OBSERVATION

RecursiveCharacterTextSplitter is smarter than simply cutting every 100th character.
Hence at every 100th character it will check for newlines spaces and understand the context before splitting the characters

## 8.3 Load Embedding Model
- Load a pre-trained embedding model.
- The model converts text chunks into numerical vectors.
- These vectors will later be stored in ChromaDB for semantic search.

In [9]:
# Display message indicating model loading has started
print("\nLoading Embedding Model...")

# Load the pre-trained MiniLM embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Display success message after model is loaded
print("Embedding Model Loaded Successfully")


Loading Embedding Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1572.48it/s]


Embedding Model Loaded Successfully


## 8.4: Generate Embeddings

- Convert every text chunk into a vector.
- Each chunk gets one embedding.
- These vectors capture the semantic meaning of the text.

In [10]:
# Display message indicating embedding generation
print("\nGenerating Embeddings...")

# Convert every text chunk into a numerical vector
embeddings = model.encode(chunks)

# Display success message
print("Embeddings Generated Successfully")


Generating Embeddings...
Embeddings Generated Successfully


## 8.5 : Display Embedding Information

- Display information about the generated embeddings.
- Show the total number of chunks.
- Display the embedding dimension and sample values.

In [11]:
# Print section heading
print("\n" + "=" * 50)
print("EMBEDDING INFORMATION")
print("=" * 50)

# Display total number of chunks
print(f"\nTotal Chunks : {len(chunks)}")

# Display dimension of one embedding vector
print(f"Embedding Dimension : {len(embeddings[0])}")

# Display heading for sample vector values
print("\nFirst 10 values of Chunk 1 Embedding:")

# Print first 10 values of first embedding vector
print(embeddings[0][:10])#[0]=first index and [:10]=start from index 0 and go till 9th index


EMBEDDING INFORMATION

Total Chunks : 4
Embedding Dimension : 384

First 10 values of Chunk 1 Embedding:
[ 0.00850519 -0.05873139  0.00638542  0.01003875 -0.01909031 -0.07677074
  0.00952041  0.0457249  -0.0148371   0.03480785]


## Observation

1.Each document chunk is converted into a fixed 384-dimensional embedding vector using the all-MiniLM-L6-v2 model.

2.The embedding is generated from tokens (words/subwords), not from individual characters. The model first tokenizes the text, understands its context using a Transformer, and then produces a single vector representing the semantic meaning of the entire chunk.

3.The 384 dimensions do not directly separate sentences. Instead, they act as a semantic representation of each chunk. During retrieval, cosine similarity compares these vectors, allowing semantically similar chunks to be retrieved even if they do not contain the exact same words.



## 8.6 Show Chunk and Embedding Relationship

- Display every chunk with its corresponding vector.
- Verify that each chunk has its own embedding.
- Print only the first five vector values for readability.

In [12]:
# Print section heading
print("\n" + "=" * 50)
print("CHUNK AND ITS EMBEDDING")
print("=" * 50)

# Loop through every chunk
for i in range(len(chunks)):

    # Display chunk number
    print(f"\nChunk {i+1}:")

    # Display chunk text
    print(chunks[i])

    # Display heading for vector
    print("\nVector (first 5 values):")

    # Print first five values of corresponding embedding
    print(embeddings[i][:5])

    # Print separator line
    print("-" * 50)


CHUNK AND ITS EMBEDDING

Chunk 1:
Artificial Intelligence is a branch of Computer Science.

Vector (first 5 values):
[ 0.00850519 -0.05873139  0.00638542  0.01003875 -0.01909031]
--------------------------------------------------

Chunk 2:
Machine Learning is a subset of Artificial Intelligence.

Vector (first 5 values):
[-0.02345927 -0.01058115  0.0719282   0.03042552  0.0285595 ]
--------------------------------------------------

Chunk 3:
Deep Learning is a subset of Machine Learning.

Neo4j is a graph database.

Vector (first 5 values):
[-0.0140926  -0.04406967 -0.0074902  -0.01593287 -0.02844735]
--------------------------------------------------

Chunk 4:
GraphRAG combines graph databases and large language models.

Vector (first 5 values):
[-0.02375095 -0.00016788 -0.02118964 -0.01151298 -0.01804933]
--------------------------------------------------


## 8.7 Connect to ChromaDB

- Create a connection with the vector database.
- Create a collection if it does not already exist.
- The collection stores chunks and embeddings.

In [15]:
# Print section heading
print("\n" + "=" * 50)
print("CONNECTING TO CHROMADB")
print("=" * 50)

# Connect to local persistent ChromaDB database
client = chromadb.PersistentClient(path="./chroma_db")

# Create the collection if it does not exist to store the document chunks along with their embeddings
collection = client.get_or_create_collection(
    name="documents"#here we are creating a collection in ChromaDB
)

# Display success messages
print("ChromaDB Connected Successfully")
print("Collection Ready")


CONNECTING TO CHROMADB
ChromaDB Connected Successfully
Collection Ready


## 8.8 Stores Chunks and Embeddings

The chunks created are now stored in the ChromaDB database



In [16]:
print("\nStoring Data in ChromaDB...")

for i, chunk in enumerate(chunks):
#adding the chunk ids,chunk ,and its embeddings in the collection
    collection.add(
        ids=[f"chunk_{i}"],#helps in inserting values inside the string
        documents=[chunk],
        embeddings=[embeddings[i].tolist()]
    )

print("All Chunks Stored Successfully")


Storing Data in ChromaDB...
All Chunks Stored Successfully


## 8.9 Verify Storage

- Verifies that all document chunks have been successfully stored in ChromaDB.
- Counts the total number of stored records in the `documents` collection.
- Retrieves the stored records from the vector database.
- Displays the unique chunk IDs for verification.
- Prints the stored document chunks to ensure correct indexing.
- Confirms that the ingestion pipeline completed successfully.



In [17]:
print("\n" + "=" * 50)
print("VERIFYING CHROMADB STORAGE")
print("=" * 50)

print(f"Total Records Stored: {collection.count()}")

# Fetch first few records

results = collection.get()

print("\nStored IDs:")
print(results["ids"])

print("\nStored Documents:")

for doc in results["documents"]:
    print("-" * 30)
    print(doc)


VERIFYING CHROMADB STORAGE
Total Records Stored: 4

Stored IDs:
['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3']

Stored Documents:
------------------------------
Artificial Intelligence is a branch of Computer Science.
------------------------------
Machine Learning is a subset of Artificial Intelligence.
------------------------------
Deep Learning is a subset of Machine Learning.

Neo4j is a graph database.
------------------------------
GraphRAG combines graph databases and large language models.


## 9: Entity Extraction from Document(03_entity_extractor.py)

1.Read the input document from the local file system.

2.Display the original document for verification.

3.Extract potential named entities using Regular Expressions (Regex).

4.Remove duplicate entities to obtain a unique list.

5.Display the extracted entities for validation.

6.The extracted entities will later be used to build the Knowledge Graph in Neo4j.

## 9.1 Import Required Library

1.Import Python's built-in re module.

2.The re module provides support for Regular Expressions.

3.Regular Expressions are used to search for patterns within text.

4.In this project, Regex is used as a simple rule-based entity extractor.

In [18]:
#Import Python's Regular Expression library
import re

## 9.2 Read the Document
1.Open the document stored on disk.

2.Read the complete document into memory.

3.Store the document as a single string.

4.Display the document before processing.

5.This text will be used for entity extraction.

In [19]:
# Open the text file in read mode
with open(
    "sample.txt",
    "r",
    encoding="utf-8"#checks for the context of the sentence
) as file:

    # Read the entire document into a string
    text = file.read()

# Display the loaded document
print("\nDOCUMENT:")
print("=" * 50)
print(text)


DOCUMENT:
Artificial Intelligence is a branch of Computer Science.

Machine Learning is a subset of Artificial Intelligence.

Deep Learning is a subset of Machine Learning.

Neo4j is a graph database.

GraphRAG combines graph databases and large language models.



## 9.3: Entity Extraction
1.Extract potential entities from the document using Regular Expressions (Regex).

2.Identify words that begin with a capital letter.

3.Store all matching words in a list.

4.Remove duplicate entities to create a unique entity list.
Display the extracted entities for verification before graph creation.

In [20]:
# Find all words that start with a capital letter
entities = re.findall(
    r'\b[A-Z][a-zA-Z]+\b',
    text
)

# Remove duplicate entities by converting the list to a set
# and then converting it back to a list
entities = list(set(entities))

# Display a heading
print("\nENTITIES FOUND")
print("=" * 50)
for entity in entities:#the probable entities are printed
    print(entity)


ENTITIES FOUND
Intelligence
Deep
Learning
GraphRAG
Artificial
Science
Machine
Computer


## 10 Store Extracted Entities into Neo4j

1.Import the required libraries for entity extraction and graph database operations.

2.Establish a connection with the Neo4j graph database.

3.Read the input document containing textual information.

4.Extract unique entities from the document using Regular Expressions.

5.Create graph nodes for each extracted entity in Neo4j.

6.Close the database connection after successful storage.

## 10.1  Import Required Libraries

1.Import Python's Regular Expression (re) module.

2.Import the Neo4j Python driver.

3.These libraries provide the functionality for entity extraction and graph database communication.

In [21]:
# Import Regular Expression library
import re

# Import Neo4j Python Driver
from neo4j import GraphDatabase

## 10.2 Connect to Neo4j
1.Define the connection credentials.

2.Create a Neo4j driver object.

3.Establish communication with the graph database.

4.Verify that the connection has been created successfully.


In [23]:


# Neo4j Server URI where bolt=protocol and the URI is the ip address of the neo4j server
URI = "bolt://172.26.0.1:7687"

# Neo4j Username
USERNAME = "neo4j"

# Neo4j Password
PASSWORD = "debishapaul16"

# Create a connection driver using the credentials
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

# Display a success message
print("Connected to Neo4j")

Connected to Neo4j


## 10.3 and 10.4 Reusing Extracted Entities

- The document has already been read in the previous step.
- Unique entities have already been extracted using Regular Expressions.
- To avoid duplication, the same entity list is reused in this step.
- The next task is to store these entities as nodes in Neo4j.

## 10.5 Prepare the Cypher Query

1.Define the Cypher query used to create graph nodes.

2.Use the MERGE clause to prevent duplicate nodes.

3.Parameterize the query for safe and efficient execution.


In [24]:
# Cypher query to create entity nodes
query = """
MERGE (e:Entity {name:$name}) 
"""#$name when value is supplied from python

## 10.6  Store Entities into Neo4j

1.Open a database session.

2.Iterate through all extracted entities.

3.Execute the Cypher query for each entity.

4.Create graph nodes inside Neo4j.

In [25]:
# Open a Neo4j session
with driver.session(database="gdb") as session:

    # Iterate through every extracted entity
    for entity in entities:

        # Execute the Cypher query
        session.run(
            query,
            name=entity
        )

print("\nEntities Stored Successfully")
driver.close()#close the driver for communication between neo4j and python


Entities Stored Successfully


## Observation
1.Neo4j Graph Storage

2.The extracted entities were successfully stored as nodes in the Neo4j Knowledge Graph.

3.The MERGE clause ensured that duplicate nodes were not created, even if an entity appeared multiple times.

4.Each entity is stored with the label Entity and the property name.
The graph can be explored interactively using Neo4j Browser or Neo4j Desktop.

Note: This step requires a running Neo4j instance. If Neo4j is not running or the connection credentials are incorrect, the notebook will raise a connection error. Users cloning this project should start Neo4j with their credentials and then update this document  before executing this section.





## 11 : Relationship Extraction and Graph Construction(05_graph_builder.py)
1.Connect to the Neo4j database.

2.Load the Gemma Large Language Model (LLM).

3.Read the input document.

4.Prompt the LLM to extract entities and relationships in JSON format.

5.Clean the generated JSON response.

6.Convert the JSON string into Python objects.

7.Create graph relationships in Neo4j using Cypher queries.

8.Close the database connection after successful graph construction.

## 11.1  Import Required Libraries

1.Import the required Python libraries.

2.Each library performs a specific task such as LLM communication, graph database operations, JSON processing, and text cleaning.

In [26]:
# Import Ollama interface to communicate with Gemma
from langchain_ollama import OllamaLLM

# Import Neo4j driver
from neo4j import GraphDatabase

# Import JSON library
import json

# Import Regular Expression library
import re

## 11.2 Connect to Neo4j

1.Define the database credentials.

2.Create a Neo4j driver.

3.Establish communication with the graph database.(same as done above in step 10)

## 11.3 : Load the Gemma LLM

1.Load the Gemma model through Ollama.

2.The model will read the document.

3.It will identify semantic relationships between entities.

In [27]:
# Load Gemma model using Ollama
llm = OllamaLLM(
    model="gemma3"
)

# Verify successful loading
print("Gemma Loaded")

Gemma Loaded


## 11.4 : Read the Document

Open the text document.
Read the complete contents.
Store it as a string for LLM processing.

(same as done earlier)

## 11.5 : Create the Prompt

1.Build an instruction for the LLM.

2.Specify the required output format.

3.Ask the model to return only JSON.

In [28]:
# Prompt instructing Gemma to extract relationships
prompt = f"""
Extract all entities and relationships.

Return ONLY JSON.

Format:

[
  {{
    "source":"Machine Learning",
    "relationship":"SUBSET_OF",
    "target":"Artificial Intelligence"
  }}
]

Text:

{text}
"""

## 11.6 Generate Relationships using Gemma

1.Send the prompt to Gemma.

2.Receive the generated JSON response.

3.Display the raw response

In [29]:


# Send prompt to Gemma
response = llm.invoke(prompt)

# Display raw output
print("\nRAW RESPONSE:")
print(response)


RAW RESPONSE:
```json
[
  {
    "source": "Artificial Intelligence",
    "relationship": "BRANCH_OF",
    "target": "Computer Science"
  },
  {
    "source": "Machine Learning",
    "relationship": "SUBSET_OF",
    "target": "Artificial Intelligence"
  },
  {
    "source": "Deep Learning",
    "relationship": "SUBSET_OF",
    "target": "Machine Learning"
  },
  {
    "source": "Neo4j",
    "relationship": "IS_A",
    "target": "graph database"
  },
  {
    "source": "GraphRAG",
    "relationship": "COMBINES",
    "target": "graph databases"
  },
  {
    "source": "GraphRAG",
    "relationship": "COMBINES",
    "target": "large language models"
  }
]
```


## 11.7 : Clean the JSON Response

1.Remove Markdown formatting.

2.Keep only the JSON data.

3.Prepare the response for parsing.

In [30]:
# Remove Markdown JSON tag
response = re.sub(
    r"```json",
    "",
    response
)

# Remove closing Markdown tag
response = re.sub(
    r"```",
    "",
    response
)

## Observation

Gamma often returns in this format which is not acceptable by python for parsing hence the wrappers are removed first
```json
[
   ...
]
```

## 11.8 : Convert JSON to Python Objects

1.Parse the JSON string.

2.Convert it into Python dictionaries.

3.Display all extracted relationships.

In [31]:
# Convert JSON string into Python list
relationships = json.loads(response)

print("\nRelationships Found:\n")

# Display every extracted relationship
for rel in relationships:
    print(rel)


Relationships Found:

{'source': 'Artificial Intelligence', 'relationship': 'BRANCH_OF', 'target': 'Computer Science'}
{'source': 'Machine Learning', 'relationship': 'SUBSET_OF', 'target': 'Artificial Intelligence'}
{'source': 'Deep Learning', 'relationship': 'SUBSET_OF', 'target': 'Machine Learning'}
{'source': 'Neo4j', 'relationship': 'IS_A', 'target': 'graph database'}
{'source': 'GraphRAG', 'relationship': 'COMBINES', 'target': 'graph databases'}
{'source': 'GraphRAG', 'relationship': 'COMBINES', 'target': 'large language models'}


## 11.9 : Store Relationships in Neo4j

1.Open a Neo4j session.

2.Iterate through each extracted relationship.

3.Create missing nodes if required.

4.Create graph edges connecting the entities.

In [33]:


with driver.session(database="gdb") as session:
    
#for getting all the relationships in the format of source relationships and target
    for rel in relationships:

        source = rel["source"]
        relation = rel["relationship"]
        target = rel["target"]
        
#Create the source node if it doesn't already exist.
#Create the target node if it doesn't already exist.
#Create the relationship between them if it doesn't already exist.
        query = f"""
        MERGE (a:Entity {{name:$source}})
        MERGE (b:Entity {{name:$target}})
        MERGE (a)-[:{relation}]->(b)
        """

        session.run(
            query,
            source=source,
            target=target
        )

print("\nGraph Created Successfully")


DriverError: Driver closed

## 12 : Retrieve Information from the Knowledge Graph

1.Connect to the Neo4j Knowledge Graph.

2.Accept an entity name from the user.

3.Execute a Cypher query to search for related nodes.

4.Display the retrieved relationships in a readable format.

5.Close the database connection after the search is complete.

## 12.1 Connect to Neo4j & Take User Input

In [ ]:
# ==================================================
# STEP 12.1 : CONNECT TO NEO4J & TAKE USER INPUT
# ==================================================

# Define the Neo4j server URI
URI = "bolt://172.26.0.1:7687"

# Define the Neo4j username
USERNAME = "neo4j"

# Define the Neo4j password
PASSWORD = "your_password"

# Create a connection (driver) to the Neo4j database
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

# Verify that the connection is successful
print("Connected To Neo4j")

# Ask the user to enter an entity to search in the graph
entity = input("Enter Entity: ")

## 12.2 Create the Cypher Query
1.Define a Cypher query to search the Knowledge Graph.

2.Match all relationships where the entered entity is the destination node.

3.Return the source node, relationship type, and target node.

In [ ]:
# ==================================================
# STEP 12.2 : CREATE CYPHER QUERY
# ==================================================

# Cypher query used to search the Knowledge Graph
query = """

# Match every pattern of the form (a)-[r]->(b)
MATCH (a)-[r]->(b)

# Compare the target node name with the user input (case-insensitive)
WHERE toLower(b.name) = toLower($entity)

# Return the source node name
RETURN
a.name AS source,

# Return the relationship type
type(r) AS relationship,

# Return the destination node name
b.name AS target

"""